In [0]:
from pyspark.sql.functions import col, when, to_date, date_format
from pyspark.sql.types import DateType

def standardize_date_column(df, column_name):
    return df.withColumn(
        column_name,
        when(
            col(column_name).contains("/"),
            date_format(to_date(col(column_name), "M/d/yyyy"), "yyyy-MM-dd")
        ).when(
            col(column_name).contains("-"),
            date_format(to_date(col(column_name), "MM-dd-yyyy"), "yyyy-MM-dd")
        ).otherwise(
            date_format(to_date(col(column_name), "yyyy-MM-dd"), "yyyy-MM-dd")
        )
    ).withColumn(
        column_name, col(column_name).cast(DateType())
    )

def lowercase_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower())
    return df

import re

def to_snake_case(df):
    def convert(col_name):
        # Replace spaces & special chars with underscore
        col_name = re.sub(r'[^a-zA-Z0-9]', '_', col_name)
        
        # Convert CamelCase → snake_case
        col_name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col_name)
        
        # Convert to lowercase
        col_name = col_name.lower()
        
        # Remove multiple underscores
        col_name = re.sub(r'_+', '_', col_name)
        
        # Remove leading/trailing underscores
        col_name = col_name.strip('_')
        
        return col_name

    for c in df.columns:
        df = df.withColumnRenamed(c, convert(c))
    
    return df

# Transforming the stores table then store it to the silver layer

In [0]:
from pyspark.sql.functions import col, when, lit, to_date, date_format

df = spark.table("01_prod_bronze.raw.stores")
df=standardize_date_column(df,"Open_Date")

# df = df.withColumn("Open_Date", to_date(col("Open_Date"), "yyyy-MM-dd"))

                 
df=lowercase_columns(df)
# display(df)
null_count_filter = df.filter(col("square_meters").isNull()).count()

df=df.fillna({"square_meters":0})

store_df=to_snake_case(df)
store_df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true")  \
  .saveAsTable("02_prod_silver.transform.stores")
display(store_df)


# 1. Check for missing/null values in Square_Meters and Open_Date

# Transforming the sales table then store it to the silver layer

In [0]:
from pyspark.sql.functions import col, when, upper
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
from pyspark.sql.functions import first

df = spark.table("01_prod_bronze.raw.sales")

# LOWERCASE
df = lowercase_columns(df)

# REMOVE INVALID ORDERS 
df = df.filter(col("order_number").isNotNull())

# HANDLE '-' in delivery_date
df = df.withColumn(
    "delivery_date",
    when(col("delivery_date") == "-", None)
    .otherwise(col("delivery_date"))
)

# STANDARDIZE DATES
df = standardize_date_column(df, "delivery_date")
df = standardize_date_column(df, "order_date")

# CAST TO DATE
df = df.withColumn("delivery_date", col("delivery_date").cast(DateType()))
df = df.withColumn("order_date", col("order_date").cast(DateType()))

# STANDARDIZE CURRENCY
df = df.withColumn("currency_code", upper(col("currency_code")))

# VALIDATE QUANTITY
df = df.filter(col("quantity") > 0)

# REMOVE DUPLICATES
df = df.dropDuplicates(["order_number", "line_item"])

# FILL customerkey USING ORDER
window_spec = Window.partitionBy("order_number")

df = df.withColumn(
    "customerkey",
    first("customerkey", ignorenulls=True).over(window_spec)
)

# df = df.withColumn(
#     "delivery_date",
#     when(col("delivery_date").isNull(), col("order_date"))
#     .otherwise(col("delivery_date"))
# )
df = df.withColumn(
    "delivery_date",
    when(col("storekey") != 0, None)  # store purchase
    .otherwise(col("delivery_date"))  # online delivery
)

# HANDLE remaining NULLS

df = df.withColumn(
    "customerkey",
    when(col("customerkey").isNull(), -1).otherwise(col("customerkey"))
)

df = df.filter(col("quantity") > 0)

# sales_df_cleaned = df.filter(col("customerkey").isNotNull())
sales_df_cleaned = df.filter(
    (col("delivery_date").isNull()) |
    (col("delivery_date") >= col("order_date"))
)

# sales_df_cleaned.filter(col("order_number") == 366005).show()
# SAVE
sales_df_cleaned=to_snake_case(sales_df_cleaned)
sales_df_cleaned.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("02_prod_silver.transform.sales")



display(sales_df_cleaned)

In [0]:
sales_df_cleaned.filter(col("order_number").isNull()).show()

# Transforming the stores table then customers it to the silver layer

In [0]:
from pyspark.sql.functions import col, when, first, upper, trim, create_map, lit, concat
from pyspark.sql.window import Window
from itertools import chain

# Read data
df = spark.table("01_prod_bronze.raw.customers")

# Lowercase columns
df = lowercase_columns(df)

# Standardize date
df = standardize_date_column(df, "birthday")

# Ensure zip_code is STRING
df = df.withColumn("zip_code", col("zip_code").cast("string"))

# -------------------------------
# Standardize BEFORE processing
# -------------------------------
df = df.withColumn("city", upper(trim(col("city"))))
df = df.withColumn("state", upper(trim(col("state"))))

# -------------------------------
#  Window fill
# -------------------------------
window_spec = Window.partitionBy("country", "city")

df = df.withColumn("state", first("state", ignorenulls=True).over(window_spec))
df = df.withColumn("state_code", first("state_code", ignorenulls=True).over(window_spec))
df = df.withColumn("zip_code", first("zip_code", ignorenulls=True).over(window_spec))

# -------------------------------
# SELF LOOKUP (STATE)
# -------------------------------
state_ref = df.select("state_code", "state") \
    .filter(col("state").isNotNull()) \
    .dropDuplicates()

df = df.join(state_ref.withColumnRenamed("state", "ref_state"),
             on="state_code", how="left")

df = df.withColumn(
    "state",
    when(col("state").isNull(), col("ref_state"))
    .otherwise(col("state"))
).drop("ref_state")

# -------------------------------
# SELF LOOKUP (ZIP CODE)
# -------------------------------
zip_ref = df.select("city", "state_code", "zip_code") \
    .filter(col("zip_code").isNotNull()) \
    .dropDuplicates()

df = df.join(zip_ref.withColumnRenamed("zip_code", "ref_zip"),
             on=["city", "state_code"], how="left")

df = df.withColumn(
    "zip_code",
    when(col("zip_code").isNull(), col("ref_zip"))
    .otherwise(col("zip_code"))
).drop("ref_zip")

# -------------------------------
#  FINAL FALLBACK (only missing cases)
# -------------------------------
zip_mapping = {
    ("GLADSTONE", "NSW"): "2440",
    ("MENINGIE EAST", "SA"): "5264",
    ("TALAROO", "QLD"): "4871",
    ("WARRANWOOD", "VIC"): "3134"
}

#  FIXED: use concat instead of +
df = df.withColumn(
    "city_state",
    concat(col("city"), lit("_"), col("state_code"))
)

# build map
mapping_list = []
for (city, state), zip_code in zip_mapping.items():
    mapping_list.extend([lit(city + "_" + state), lit(zip_code)])

zip_expr = create_map(mapping_list)

# apply fallback
df = df.withColumn(
    "zip_code",
    when(col("zip_code").isNull(), zip_expr[col("city_state")])
    .otherwise(col("zip_code"))
).drop("city_state")

customer_df = df.withColumn(
    "customerkey",
    when(col("customerkey").isNull(), -1).otherwise(col("customerkey"))
)


# df=df.filter(col("customerkey").isNotNull())
# -------------------------------
# Write to Silver
# -------------------------------
customer_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("02_prod_silver.transform.customers")

# -------------------------------
# Validation Checks
# -------------------------------
# print("Remaining NULL state_code:")
# df.filter(col("state_code").isNull()).show()

# print("Remaining NULL zip_code:")
# df.filter(col("zip_code").isNull()).show()

display(customer_df)

# Transforming the products table then store it to the silver layer

In [0]:
from pyspark.sql.functions import regexp_replace, col

from pyspark.sql.functions import when

df = spark.table("01_prod_bronze.raw.products")
df=lowercase_columns(df)
df = df.withColumn(
    "unit_cost_usd",
    regexp_replace(col("unit_cost_usd"), "[$,]", "").cast("double")
)

df = df.withColumn(
    "unit_price_usd",
    regexp_replace(col("unit_price_usd"), "[$,]", "").cast("double")
)

df = df.withColumn(
    "product_name",
    regexp_replace(col("product_name"), '"', "")
)

from pyspark.sql.functions import upper, trim

df = df.withColumn("brand", upper(trim(col("brand")))) \
       .withColumn("color", upper(trim(col("color")))) \
       .withColumn("subcategory", upper(trim(col("subcategory")))) \
       .withColumn("category", upper(trim(col("category"))))



product_df = df.withColumn(
    "subcategory",
    when(col("subcategory") == "MP4&MP3", "MP4 & MP3")
    .otherwise(col("subcategory"))
)
product_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("02_prod_silver.transform.products")
display(product_df)



# Transforming the exchange_rate table then store it to the silver layer

In [0]:
from pyspark.sql.functions import upper

df = spark.table("01_prod_bronze.raw.exchange_rate")
df=standardize_date_column(df,"Date")
df=lowercase_columns(df)


df = df.withColumn("currency", upper(col("currency")))
df = df.withColumn("exchange", col("exchange").cast("double"))

df=df.withColumnRenamed("currency","currency_code")
exchange_rate_df=df.withColumnRenamed("exchange","exchange_rate")
exchange_rate_df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("02_prod_silver.transform.exchange_rate")

display(exchange_rate_df)